In [6]:
import torch
from torch import nn
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import sentencepiece as sp

In [7]:
vocab_size = 1024
embed_size = 256
num_heads = 8
ff_size = 1024
num_blocks = 8
context_size = 512
batch_size = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [8]:
sp_model = sp.SentencePieceProcessor(model_file='gutenberg_poetry_tokenizer.model')

In [4]:
random.seed(777)

class DataLoader():
    def __init__(self, text_file: str, tokenizer_model: sp.SentencePieceProcessor, context_size: int, val_percentage: float):
        text_corpus = None
        with open(text_file, 'r') as f:
            text_corpus = f.read()

        data = torch.tensor(tokenizer_model.encode_as_ids(text_corpus))
        num_chunks = len(data) // context_size
        if len(data) % context_size == 0:
            num_chunks -= 1
        chunks = []
        for i in range(num_chunks):
            chunk = data[i : i + context_size + 1]
            x = chunk[:-1]
            y = chunk[1:]
            chunks.append(torch.stack((x, y)))

        random.shuffle(chunks)

        num_val_chunks = int(val_percentage * num_chunks)

        self.val_chunks = chunks[:num_val_chunks]
        self.train_chunks = chunks[num_val_chunks:]


    def train_data(self, batch_size: int):
        train_chunks = [
            torch.stack(self.train_chunks[i:i+batch_size]).unbind(-2) for i in range(0, len(self.train_chunks), batch_size)
            ]
        return train_chunks

    def val_data(self, batch_size: int):
        val_chunks = [
            torch.stack(self.val_chunks[i:i+batch_size]).unbind(-2) for i in range(0, len(self.val_chunks), batch_size)
            ]
        return val_chunks


data_loader = DataLoader('./data/Gutenberg-Poetry.txt',
                         sp_model,
                         context_size,
                         0.125)

In [9]:
class RoPE(nn.Module):
    def __init__(self, d_head, s_length_max, base_freq=1024):
        super(RoPE, self).__init__()

        self.d_head = d_head

        inv_freq = 1.0 / (base_freq ** (torch.arange(0, d_head, 2) / d_head))
        pos = torch.arange(0, s_length_max, dtype=torch.float)
        angle = torch.outer(pos, inv_freq)

        self.register_buffer("cos", torch.cos(angle), persistent=False)
        self.register_buffer("sin", torch.sin(angle), persistent=False)

    def forward(self, x):
        s = x.size(-2) # context size
        d = self.d_head

        x_ = x.view(*x.shape[:-1], d // 2, 2)
        x1 = x_[..., 0]
        x2 = x_[..., 1]

        cos = self.cos[:s].to(dtype=x.dtype, device=x.device)
        sin = self.sin[:s].to(dtype=x.dtype, device=x.device)

        y1 = x1 * cos - x2 * sin
        y2 = x1 * sin + x2 * cos

        y = torch.stack((y1, y2), dim=-1).view_as(x)

        return y


class Attention(nn.Module):
    def __init__(self):
        super(Attention, self).__init__()

        self.head_size = embed_size // num_heads
        self.rope = RoPE(self.head_size, context_size, 1024)
        self.qkv_linear = nn.Linear(embed_size, 3 * embed_size, bias=False)
        self.heads_fuser = nn.Linear(embed_size, embed_size, bias=False)

        self.register_buffer("mask", torch.triu(torch.full((context_size, context_size), -torch.inf), 1), persistent=False)


    def forward(self, x: torch.Tensor):
        x_ = self.qkv_linear(x)

        x_ = x_.view(*x_.shape[:2], 3, num_heads, self.head_size).transpose(1, 3)

        q, k, v = x_.unbind(dim=2)

        q, k = self.rope(q), self.rope(k)

        s = q @ k.transpose(-1, -2) / math.sqrt(self.head_size)

        mask = self.mask[:x.size(1), :x.size(1)].to(dtype=x.dtype, device=x.device)
        s += mask
        s = torch.softmax(s, -1)
        # DROPOUT
        a = s @ v
        a = a.transpose(1,2).reshape_as(x)

        y = self.heads_fuser(a)

        return y


class SwiGLU(nn.Module):
    def __init__(self):
        super(SwiGLU, self).__init__()

        self.in_layer = nn.Linear(embed_size, ff_size * 2, bias=False)
        self.silu = nn.SiLU()
        self.out_layer = nn.Linear(ff_size, embed_size, bias=False)

    def forward(self, x: torch.Tensor):
        ab = self.in_layer(x)
        a, b = ab.chunk(2, -1)

        y = a * self.silu(b)
        y = self.out_layer(y)

        return y


"""
                ____________
                |           x
                |           |
                |       RMSNorm
                |           |___________
                |           KQV         |
                |           |           |
                |   Multihead split     |
                |           |           |
                |       RoPE(Q,K)       |
                |           |           |   Attention
Attention Block |   Attention(K,Q,V)    |
                |           |           |
                |       Fuse Heads      |
                |           |           |
                |       x + |___________|
                |           |
                |       RMSNorm
                |           |
                |       SwiGLU FFN
                |           |
                |_______x + |
"""
class AttentionBlock(nn.Module):
    def __init__(self):
        super(AttentionBlock, self).__init__()

        self.pre_norm = nn.RMSNorm(embed_size)
        self.attention = Attention()
        self.post_norm = nn.RMSNorm(embed_size)
        self.swiglu = SwiGLU()


    def forward(self, x: torch.Tensor):
        y = self.attention(self.pre_norm(x))
        y = self.swiglu(self.post_norm(x + y))
        return x + y


class AttentionBlocksStack(nn.Module):
    def __init__(self, num_blocks):
        super(AttentionBlocksStack, self).__init__()

        self.atten_blocks = nn.ModuleList([AttentionBlock() for _ in range(num_blocks)])

    def forward(self, x: torch.Tensor):
        for atten_block in self.atten_blocks:
            x = atten_block(x)

        return x


class Rockformer(nn.Module):
    def __init__(self):
        super(Rockformer, self).__init__()

        self.embed = nn.Embedding(vocab_size, embed_size)

        self.atten_blocks = AttentionBlocksStack(num_blocks)
        self.post_norm = nn.RMSNorm(embed_size)
        self.reverse_embed = nn.Linear(embed_size, vocab_size, bias=False)

        nn.init.normal_(self.embed.weight, mean=0.0, std=0.02)
        self.reverse_embed.weight = self.embed.weight

        self.seq = nn.Sequential(
                                    self.embed,
                                    self.atten_blocks,
                                    self.post_norm,
                                    self.reverse_embed
                                )

    def forward(self, x):
        return self.seq(x)

In [ ]:
torch.manual_seed(777)

model = Rockformer().to(device).train()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001, weight_decay=0.1)
cross_entropy = nn.CrossEntropyLoss()

epochs = 5
train_data = data_loader.train_data(batch_size)
eval_data = data_loader.val_data(batch_size)
for epoch in range(epochs):
    for i, (x, y) in enumerate(train_data):
        with torch.autocast("cuda", dtype=torch.bfloat16):
            y_pred = model(x.to(device))
            loss = cross_entropy(y_pred.view(-1, vocab_size), y.reshape(-1).to(device=device))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if i % 100 == 0:
            print(f"\tLoss_{i:05d}: {loss.cpu().detach().item()}")

    print(f"Epoch #{epoch}:\nTrain Loss: {loss.cpu().detach().item()}")
    model = model.eval()
    with torch.no_grad():
        losses = []
        for x, y in eval_data:
            with torch.autocast("cuda", dtype=torch.bfloat16):
                y_pred = model(x.to(device))
                loss = cross_entropy(y_pred.view(-1, vocab_size), y.reshape(-1).to(device=device)).cpu().detach().item()
                losses.append(loss)

    print(f"Epoch #{epoch}:\nEval Average Loss: {sum(losses)/len(losses)}")

    checkpoint = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epoch
        }

    torch.save(checkpoint, f"checkpoint_epoch_{epoch:03d}.pt")
    model = model.train()


	Loss_00000: 6.975028991699219
	Loss_00100: 5.704258918762207
	Loss_00200: 4.5726094245910645
	Loss_00300: 3.762864589691162
	Loss_00400: 2.933882713317871
	Loss_00500: 2.225144147872925
	Loss_00600: 1.6925089359283447
	Loss_00700: 1.1876487731933594
	Loss_00800: 0.8245213031768799
	Loss_00900: 0.6193891763687134
	Loss_01000: 0.4062410891056061
	Loss_01100: 0.322819322347641
	Loss_01200: 0.22351142764091492
Epoch #0:
Train Loss: 0.24150221049785614
Epoch #0:
Eval Average Loss: 0.2194964567991509
	Loss_00000: 0.22321945428848267
	Loss_00100: 0.17362579703330994
	Loss_00200: 0.14245155453681946
	Loss_00300: 0.1230911910533905
	Loss_00400: 0.11559361219406128
	Loss_00500: 0.09954270720481873
	Loss_00600: 0.08927047252655029
	Loss_00700: 0.08293057978153229
	Loss_00800: 0.07518601417541504
	Loss_00900: 0.08035627007484436
	Loss_01000: 0.06839068979024887
	Loss_01100: 0.0682312399148941
	Loss_01200: 0.06007195636630058
Epoch #1:
Train Loss: 0.0649922713637352
Epoch #1:
Eval Average Loss: 0.

In [ ]:
checkpoint = torch.load("checkpoint_epoch_004.pt", map_location='cuda')

gen_model = Rockformer().eval().to(device)
gen_model.load_state_dict(checkpoint["model_state_dict"])

initial_str = "Hello there! What is your name?"
tokens = torch.tensor(sp_model.encode_as_ids(initial_str)).unsqueeze(0).to(device)

with torch.no_grad():
    for _ in range(128):
        last_token_logits = gen_model(tokens)[:,-1]
        new_token_id = torch.argmax(last_token_logits, dim=-1).unsqueeze(0)
        tokens = torch.cat((tokens, new_token_id), dim=-1)

print(sp_model.decode_ids(tokens.cpu().tolist()))

['Hello there! What is your name? O Birch-tree! Of your yellow bark, O Birch-tree! Growing by the rushing river, Tall and stately in the valley! I a light canoe will build me, Build a swift Cheemaun for sailing, That shall float upon the river, Like a yellow leaf in Autumn, Like a yellow water-lily! "Lay aside your cloak, O Birch-tree! L']
